# **Fase #0:** Preparación e investigación teórica

Predicción del precio de vivienda por unidad de área
Real Estate Valuation Data Set — UCI ML Repository
UCSG — Big Data / Machine Learning — Grupo 4

# 📘 Fase 0: Investigación Teórica de Métricas de Regresión

## 1. Diferencia entre Regresión y Clasificación

En **clasificación** la variable objetivo es **categórica** (discreta), por lo que se evalúa si la clase predicha coincide con la real (accuracy, precision, recall, F1).

En **regresión** la variable objetivo es **continua** (un número real), por lo que no tiene sentido hablar de "aciertos"; en su lugar medimos **la magnitud del error** entre el valor predicho y el real.

---

## 2. Tabla Resumen de Métricas

| Métrica | Fórmula | Unidades | Qué mide | Cuándo usarla |
|---------|---------|----------|----------|---------------|
| **MAE** | (1/n) × Σ\|y - ŷᵢ\| | Mismas que Y | Error absoluto promedio | Interpretación simple, robusta a outliers |
| **MSE** | (1/n) × Σ(yᵢ - ŷᵢ)² | Unidades de Y² | Error cuadrático promedio | Penaliza errores grandes |
| **RMSE** | √MSE | Mismas que Y | Desviación típica de los residuos | Comparar modelos cuando importan los outliers |
| **MAPE** | (100/n) × Σ\|(y - ŷᵢ)/yᵢ\| | Porcentaje (%) | Error relativo medio | Comparar error en términos relativos; evitar si y=0 |
| **R²** | 1 - (SS_res/SS_tot) | Adimensional (0-1) | Proporción de varianza explicada | Comparación rápida; puede engañar con muchas variables |
| **R² Ajustado** | 1 - (1-R²)×(n-1)/(n-p-1) | Adimensional | R² penalizado por n° de predictores | Comparar modelos con distinto n° de variables |

---

## 3. Preguntas Clave

### ¿Por qué el R² por sí solo puede engañar?

Por construcción matemática, el R² **nunca disminuye** al añadir variables nuevas, aunque estas no aporten información real. Esto favorece modelos sobreajustados. El **R² ajustado** corrige este efecto penalizando el uso de predictores innecesarios.

**Ejemplo:** Si añades una variable aleatoria que no tiene relación con Y, el R² subirá ligeramente, pero el R² ajustado bajará porque penaliza el número de predictores.

---

### RMSE vs MAE con outliers

**RMSE** eleva los errores al cuadrado → **penaliza fuertemente los outliers**.
- ✅ Útil cuando un error grande es inaceptable (ej. precios muy lejanos)
- ❌ Puede ser dominado por pocos valores atípicos

**MAE** trata todos los errores linealmente → **es más robusto a outliers**.
- ✅ Preferible cuando los valores atípicos son ruido y no deberían dominar la métrica
- ❌ No distingue entre errores pequeños y grandes de forma proporcional

---

## 4. Fuentes Consultadas

- Documentación oficial de `scikit-learn`: [sklearn.metrics](https://scikit-learn.org/stable/modules/model_evaluation.html#regression-metrics)
- "An Introduction to Statistical Learning" (James et al., Capítulo 2)
- Apuntes de clase de Machine Learning/Big Data

---

> 📝 **Nota:** Todas las fórmulas están expresadas en notación matemática estándar. En la implementación usamos las funciones de `sklearn.metrics` que calculan estas métricas de forma optimizada.

# Fase #0: Preparación del entorno

In [1]:
# FASE 0: Configuración del Entorno, Estilo y Carga de Datos

# 1. Instalación de librerías necesarias
!pip install plotly xgboost openpyxl -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import warnings
warnings.filterwarnings('ignore')

# ==========================================================
# CONFIGURACIÓN GLOBAL DE ESTILO (PALETA MATLAB)
# ==========================================================
# Colores clásicos de MATLAB para gráficos categóricos (barras, líneas)
matlab_colors = ['#0072BD', '#D95319', '#EDB120', '#7E2F8E', '#77AC30', '#4DBEEE', '#A2142F']

# Configurar Matplotlib y Seaborn
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=matlab_colors)
sns.set_palette(matlab_colors)
sns.set_theme(style="whitegrid", font_scale=1.1)

# Configurar Plotly con tema "matlab"
matlab_template = go.layout.Template(
    layout=go.Layout(
        colorway=matlab_colors,
        plot_bgcolor="white",
        paper_bgcolor="white",
        font=dict(family="Arial, sans-serif", size=12, color="black"),
        xaxis=dict(showgrid=True, gridcolor="lightgray", zeroline=False, showline=True, linecolor="black"),
        yaxis=dict(showgrid=True, gridcolor="lightgray", zeroline=False, showline=True, linecolor="black")
    )
)
pio.templates["matlab"] = matlab_template
pio.templates.default = "matlab"

# Paleta "Parula" (colores de mapas de calor de MATLAB) para usar en heatmaps de Plotly
parula_colors = [
    "#3B0F70", "#461E85", "#4A2C97", "#483D9D", "#403CAA",
    "#355CB3", "#2B7AB8", "#2191BC", "#1AA6B8", "#25BBB0",
    "#42C99C", "#6CD382", "#9DD963", "#D1E753", "#FDE725"
]

# ==========================================================
# CARGA DE DATOS DESDE GITHUB (URL RAW)
# ==========================================================
# El enlace github es de la interfaz web. Para que pandas lo lea,
# debemos usar el enlace en formato RAW (texto plano).
url_raw = "https://raw.githubusercontent.com/jermrodr/BIG-DATA/33691c6877a671a47100e75be22b4441b7e55848/Real%20estate%20valuation%20data%20set.xlsx"

try:
    df = pd.read_excel(url_raw)
    print("✅ Dataset cargado correctamente desde GitHub.")
    print(f"📊 Dimensiones: {df.shape[0]} filas y {df.shape[1]} columnas.")
    display(df.head())
except Exception as e:
    print(f"❌ Error al cargar el dataset: {e}")

✅ Dataset cargado correctamente desde GitHub.
📊 Dimensiones: 414 filas y 8 columnas.


,No,X1 transaction date,X2 house age,X3 distance to the nearest MRT station,X4 number of convenience stores,X5 latitude,X6 longitude,Y house price of unit area
0,1,2012.916667,32.0,84.87882,10,24.98298,121.54024,37.9
1,2,2012.916667,19.5,306.59470,9,24.98034,121.53951,42.2
2,3,2013.583333,13.3,561.98450,5,24.98746,121.54391,47.3
3,4,2013.500000,13.3,561.98450,5,24.98746,121.54391,54.8
4,5,2012.833333,5.0,390.56840,5,24.97937,121.54245,43.1


# Fase #1: Carga y exploración de datos (EDA)

In [2]:
# ==========================================================
# FASE 1: ANÁLISIS EXPLORATORIO DE DATOS (EDA)
# ==========================================================

print("="*50)
print("1. REVISIÓN INICIAL DE DATOS")
print("="*50)
display(df.info())

print("\n--- Valores Nulos y Duplicados ---")
print(f"Valores nulos por columna:\n{df.isnull().sum()}")
print(f"\nFilas duplicadas: {df.duplicated().sum()}")

print("\n" + "="*50)
print("2. ESTADÍSTICA DESCRIPTIVA")
print("="*50)
display(df.describe())

print("\n" + "="*50)
print("3. DISTRIBUCIÓN DE LA VARIABLE OBJETIVO (Y)")
print("="*50)

# Histograma de la variable objetivo
fig_hist_y = px.histogram(
    df,
    x='Y house price of unit area',
    nbins=30,
    title='Distribución del Precio de Vivienda por Unidad de Área (Y)',
    labels={'Y house price of unit area': 'Precio (10,000 NT$/Ping)'},
    color_discrete_sequence=['#0072BD'] # Azul MATLAB
)
fig_hist_y.update_layout(template='matlab')
fig_hist_y.show()

# Boxplot de la variable objetivo
fig_box_y = px.box(
    df,
    y='Y house price of unit area',
    title='Boxplot del Precio de Vivienda por Unidad de Área (Y)',
    labels={'Y house price of unit area': 'Precio'},
    color_discrete_sequence=['#D95319'] # Naranja MATLAB
)
fig_box_y.update_layout(template='matlab')
fig_box_y.show()

print("\n" + "="*50)
print("4. MATRIZ DE CORRELACIÓN Y HEATMAP")
print("="*50)

# Calcular matriz de correlación
corr_matrix = df.corr()

# Crear la escala de colores Parula para Plotly (lista de tuplas [valor, color])
parula_scale = [[i/(len(parula_colors)-1), color] for i, color in enumerate(parula_colors)]

# Heatmap interactivo
fig_heatmap = px.imshow(
    corr_matrix,
    aspect="auto",
    color_continuous_scale=parula_scale,
    title='Matriz de Correlación entre Variables (Paleta Parula)',
    labels={'color': 'Correlación'}
)
# Añadir los valores numéricos dentro de las celdas
fig_heatmap.update_traces(texttemplate="%{z:.2f}", textfont_size=10)
fig_heatmap.update_layout(template='matlab')
fig_heatmap.show()

print("\n" + "="*50)
print("5. DETECCIÓN DE OUTLIERS EN VARIABLES PREDICTORAS")
print("="*50)

features_to_check = ['X2 house age', 'X3 distance to the nearest MRT station', 'X4 number of convenience stores']

# Boxplots de las features
fig_outliers = make_subplots(rows=1, cols=3, subplot_titles=features_to_check)

for i, feature in enumerate(features_to_check):
    fig_outliers.add_trace(
        go.Box(y=df[feature], name=feature, marker_color=matlab_colors[i]),
        row=1, col=i+1
    )

fig_outliers.update_layout(title_text="Detección de Outliers en Variables Predictoras", showlegend=False, template='matlab')
fig_outliers.show()

print("\n" + "="*50)
print("6. RELACIÓN ENTRE FEATURES Y TARGET (SCATTER PLOTS)")
print("="*50)

# Scatter plots para ver la relación con el precio
fig_scatter = make_subplots(rows=1, cols=3, subplot_titles=features_to_check)

for i, feature in enumerate(features_to_check):
    fig_scatter.add_trace(
        go.Scatter(x=df[feature], y=df['Y house price of unit area'], mode='markers',
                   marker_color=matlab_colors[i], name=feature, marker_size=6, opacity=0.7),
        row=1, col=i+1
    )

fig_scatter.update_layout(title_text="Relación entre Variables Predictoras y Precio (Y)", showlegend=False, template='matlab')
fig_scatter.update_yaxes(title_text="Precio (Y)", row=1, col=1)
fig_scatter.show()

1. REVISIÓN INICIAL DE DATOS
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 414 entries, 0 to 413
Data columns (total 8 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   No                                      414 non-null    int64  
 1   X1 transaction date                     414 non-null    float64
 2   X2 house age                            414 non-null    float64
 3   X3 distance to the nearest MRT station  414 non-null    float64
 4   X4 number of convenience stores         414 non-null    int64  
 5   X5 latitude                             414 non-null    float64
 6   X6 longitude                            414 non-null    float64
 7   Y house price of unit area              414 non-null    float64
dtypes: float64(6), int64(2)
memory usage: 26.0 KB


None


--- Valores Nulos y Duplicados ---
Valores nulos por columna:
No                                        0
X1 transaction date                       0
X2 house age                              0
X3 distance to the nearest MRT station    0
X4 number of convenience stores           0
X5 latitude                               0
X6 longitude                              0
Y house price of unit area                0
dtype: int64

Filas duplicadas: 0

2. ESTADÍSTICA DESCRIPTIVA


,No,X1 transaction date,X2 house age,X3 distance to the nearest MRT station,X4 number of convenience stores,X5 latitude,X6 longitude,Y house price of unit area
count,414.000000,414.000000,414.000000,414.000000,414.000000,414.000000,414.000000,414.000000
mean,207.500000,2013.148953,17.712560,1083.885689,4.094203,24.969030,121.533361,37.980193
std,119.655756,0.281995,11.392485,1262.109595,2.945562,0.012410,0.015347,13.606488
min,1.000000,2012.666667,0.000000,23.382840,0.000000,24.932070,121.473530,7.600000
25%,104.250000,2012.916667,9.025000,289.324800,1.000000,24.963000,121.528085,27.700000
50%,207.500000,2013.166667,16.100000,492.231300,4.000000,24.971100,121.538630,38.450000
75%,310.750000,2013.416667,28.150000,1454.279000,6.000000,24.977455,121.543305,46.600000
max,414.000000,2013.583333,43.800000,6488.021000,10.000000,25.014590,121.566270,117.500000



3. DISTRIBUCIÓN DE LA VARIABLE OBJETIVO (Y)



4. MATRIZ DE CORRELACIÓN Y HEATMAP



5. DETECCIÓN DE OUTLIERS EN VARIABLES PREDICTORAS



6. RELACIÓN ENTRE FEATURES Y TARGET (SCATTER PLOTS)


# Fase #2: Preprocesamiento

In [3]:
# ==========================================================
# FASE 2: PREPROCESAMIENTO DE DATOS
# ==========================================================

print("="*60)
print("FASE 3: PREPROCESAMIENTO Y PREPARACIÓN DE DATOS")
print("="*60)

# ----------------------------------------------------------
# 3.1. ELIMINACIÓN DE COLUMNA IDENTIFICADORA (No)
# ----------------------------------------------------------
print("\n🔹 3.1. Eliminando columna 'No' (identificador secuencial)...")
df_clean = df.drop(columns=['No']).copy()
print(f"✅ Columnas restantes: {df_clean.columns.tolist()}")

# ----------------------------------------------------------
# 3.2. LIMPIEZA DEL OUTLIER EXTREMO EN LA VARIABLE OBJETIVO (Y)
# ----------------------------------------------------------
print("\n🔹 3.2. Limpiando outlier extremo en 'Y' (precio = 117.5)...")
print(f"   Filas antes de limpiar: {len(df_clean)}")

# Identificamos y eliminamos la fila con precio > 100 (outlier evidente)
outlier_mask = df_clean['Y house price of unit area'] > 100
print(f"   Outliers detectados en Y (>100): {outlier_mask.sum()}")

df_clean = df_clean[~outlier_mask].reset_index(drop=True)
print(f"✅ Filas después de limpiar: {len(df_clean)}")

# ----------------------------------------------------------
# 3.3. INGENIERÍA DE CARACTERÍSTICAS: TRANSFORMACIÓN DE FECHA
# ----------------------------------------------------------
print("\n🔹 3.3. Transformando fecha decimal a 'meses transcurridos'...")
# La fecha está en formato decimal (ej: 2012.917).
# Convertimos a meses desde el inicio del periodo para facilitar el aprendizaje.

# Extraemos año y mes aproximado
year = df_clean['X1 transaction date'].astype(int)
month_decimal = df_clean['X1 transaction date'] - year
month = (month_decimal * 12).round().astype(int)

# Calculamos meses transcurridos desde el inicio (2012.0 = mes 0)
df_clean['X1_months_since_start'] = (year - 2012) * 12 + month

# Eliminamos la columna original de fecha
df_clean = df_clean.drop(columns=['X1 transaction date'])
print(f"✅ Nueva columna 'X1_months_since_start' creada. Rango: {df_clean['X1_months_since_start'].min()} a {df_clean['X1_months_since_start'].max()} meses")

# ----------------------------------------------------------
# 3.4. TRANSFORMACIÓN DE LA DISTANCIA AL MRT (X3)
# ----------------------------------------------------------
print("\n🔹 3.4. Aplicando transformación logarítmica a la distancia al MRT...")
# La distancia tiene una distribución muy sesgada (de 23 a 6488 metros).
# Aplicamos log1p (log(1+x)) para normalizar la distribución y suavizar outliers.

from sklearn.preprocessing import FunctionTransformer

# Guardamos la distancia original para referencia
df_clean['X3_distance_original'] = df_clean['X3 distance to the nearest MRT station']

# Aplicamos transformación logarítmica
df_clean['X3 distance to the nearest MRT station'] = np.log1p(df_clean['X3 distance to the nearest MRT station'])
print(f"✅ Distancia transformada. Rango original: 23 - 6488 m | Rango log: {df_clean['X3 distance to the nearest MRT station'].min():.2f} - {df_clean['X3 distance to the nearest MRT station'].max():.2f}")

# ----------------------------------------------------------
# 3.5. VISUALIZACIÓN ANTES vs DESPUÉS (VERIFICACIÓN)
# ----------------------------------------------------------
print("\n🔹 3.5. Verificando visualmente las transformaciones...")

fig_verify = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        "Distribución de Y (después de limpiar outlier)",
        "Distribución de Distancia MRT (transformada con log)",
        "Meses transcurridos (nueva feature)",
        "Boxplot final de todas las variables numéricas"
    ],
    specs=[[{"type": "histogram"}, {"type": "histogram"}],
           [{"type": "histogram"}, {"type": "box"}]]
)

# Y limpia
fig_verify.add_trace(
    go.Histogram(x=df_clean['Y house price of unit area'], nbinsx=25,
                 marker_color=matlab_colors[0], name='Y limpio'),
    row=1, col=1
)

# Distancia transformada
fig_verify.add_trace(
    go.Histogram(x=df_clean['X3 distance to the nearest MRT station'], nbinsx=30,
                 marker_color=matlab_colors[1], name='Distancia log'),
    row=1, col=2
)

# Meses transcurridos
fig_verify.add_trace(
    go.Histogram(x=df_clean['X1_months_since_start'], nbinsx=20,
                 marker_color=matlab_colors[2], name='Meses'),
    row=2, col=1
)

# Boxplot general
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
for i, col in enumerate(numeric_cols):
    fig_verify.add_trace(
        go.Box(y=df_clean[col], name=col[:15], marker_color=matlab_colors[i % len(matlab_colors)], showlegend=False),
        row=2, col=2
    )

fig_verify.update_layout(
    height=800,
    title_text="Verificación de Transformaciones Aplicadas",
    template='matlab'
)
fig_verify.show()

# ----------------------------------------------------------
# 3.6. SEPARACIÓN DE FEATURES (X) Y TARGET (y)
# ----------------------------------------------------------
print("\n🔹 3.6. Separando variables predictoras (X) y variable objetivo (y)...")
X = df_clean.drop(columns=['Y house price of unit area', 'X3_distance_original'])
y = df_clean['Y house price of unit area']

print(f"✅ Shape de X (features): {X.shape}")
print(f"✅ Shape de y (target): {y.shape}")
print(f"   Features finales: {X.columns.tolist()}")

# ----------------------------------------------------------
# 3.7. DIVISIÓN TRAIN / TEST (80% / 20%)
# ----------------------------------------------------------
print("\n🔹 3.7. Dividiendo datos en Train (80%) y Test (20%)...")
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,  # Semilla fija para reproducibilidad
    shuffle=True
)

print(f"✅ Train: {X_train.shape[0]} muestras | Test: {X_test.shape[0]} muestras")

# ----------------------------------------------------------
# 3.8. ESCALADO DE VARIABLES (StandardScaler)
# ----------------------------------------------------------
print("\n🔹 3.8. Aplicando escalado estándar (StandardScaler)...")
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Ajustamos el escalador SOLO con los datos de train (evita data leakage)
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convertimos a DataFrame para mantener los nombres de columnas
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns, index=X_test.index)

print("✅ Escalado completado.")
print("\n📊 Comparativa de medias y desviaciones ANTES vs DESPUÉS del escalado:")
print("\n--- ANTES (X_train original) ---")
print(X_train.describe().loc[['mean', 'std']].round(3))
print("\n--- DESPUÉS (X_train_scaled) ---")
print(X_train_scaled.describe().loc[['mean', 'std']].round(3))

# ----------------------------------------------------------
# 3.9. RESUMEN FINAL DE LA FASE 3
# ----------------------------------------------------------
print("\n" + "="*60)
print("✅ FASE 3 COMPLETADA - RESUMEN")
print("="*60)
print(f"📌 Dataset final: {len(df_clean)} registros limpios")
print(f"📌 Features utilizadas: {X.shape[1]}")
print(f"📌 Transformaciones aplicadas:")
print("   • Eliminada columna 'No' (ID)")
print("   • Eliminado 1 outlier extremo en Y (precio = 117.5)")
print("   • Fecha convertida a 'meses transcurridos'")
print("   • Distancia al MRT transformada con log1p")
print(f"📌 División: Train={len(X_train)} | Test={len(X_test)}")
print("📌 Escalado: StandardScaler aplicado")
print("\n🎯 Los datos están listos para la FASE 4: MODELADO")

FASE 3: PREPROCESAMIENTO Y PREPARACIÓN DE DATOS

🔹 3.1. Eliminando columna 'No' (identificador secuencial)...
✅ Columnas restantes: ['X1 transaction date', 'X2 house age', 'X3 distance to the nearest MRT station', 'X4 number of convenience stores', 'X5 latitude', 'X6 longitude', 'Y house price of unit area']

🔹 3.2. Limpiando outlier extremo en 'Y' (precio = 117.5)...
   Filas antes de limpiar: 414
   Outliers detectados en Y (>100): 1
✅ Filas después de limpiar: 413

🔹 3.3. Transformando fecha decimal a 'meses transcurridos'...
✅ Nueva columna 'X1_months_since_start' creada. Rango: 8 a 19 meses

🔹 3.4. Aplicando transformación logarítmica a la distancia al MRT...
✅ Distancia transformada. Rango original: 23 - 6488 m | Rango log: 3.19 - 8.78

🔹 3.5. Verificando visualmente las transformaciones...



🔹 3.6. Separando variables predictoras (X) y variable objetivo (y)...
✅ Shape de X (features): (413, 6)
✅ Shape de y (target): (413,)
   Features finales: ['X2 house age', 'X3 distance to the nearest MRT station', 'X4 number of convenience stores', 'X5 latitude', 'X6 longitude', 'X1_months_since_start']

🔹 3.7. Dividiendo datos en Train (80%) y Test (20%)...
✅ Train: 330 muestras | Test: 83 muestras

🔹 3.8. Aplicando escalado estándar (StandardScaler)...
✅ Escalado completado.

📊 Comparativa de medias y desviaciones ANTES vs DESPUÉS del escalado:

--- ANTES (X_train original) ---
      X2 house age  X3 distance to the nearest MRT station  \
mean        17.191                                   6.425   
std         11.219                                   1.108   

      X4 number of convenience stores  X5 latitude  X6 longitude  \
mean                            4.082       24.969       121.533   
std                             2.957        0.013         0.015   

      X1_months_sinc

# Fase #3: Modelado

In [4]:
# ==========================================================
# FASE 4: MODELADO Y OPTIMIZACIÓN DE HIPERPARÁMETROS
# ==========================================================

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV
import time

print("="*60)
print("FASE 4: ENTRENAMIENTO Y OPTIMIZACIÓN DE MODELOS")
print("="*60)

# Diccionario para guardar los modelos entrenados y sus predicciones
modelos_entrenados = {}
predicciones_test = {}

# ----------------------------------------------------------
# 4.1. MODELO 1: REGRESIÓN LINEAL MÚLTIPLE (BASELINE)
# ----------------------------------------------------------
print("\n🔹 4.1. Entrenando Regresión Lineal Múltiple (Baseline)...")
inicio = time.time()

lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

# Guardar modelo y predicciones
modelos_entrenados['Regresión Lineal'] = lr_model
predicciones_test['Regresión Lineal'] = lr_model.predict(X_test_scaled)

fin = time.time()
print(f"✅ Regresión Lineal entrenada en {fin-inicio:.4f} segundos.")
print(f"   (Nota: La regresión lineal no tiene hiperparámetros para tunear).")

# ----------------------------------------------------------
# 4.2. MODELO 2: RANDOM FOREST CON GRIDSEARCHCV
# ----------------------------------------------------------
print("\n🔹 4.2. Entrenando Random Forest con Validación Cruzada (GridSearch)...")
inicio = time.time()

# Definir la grilla de hiperparámetros (mantenida pequeña para no tardar mucho en Colab)
param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 5, 10, 15],
    'min_samples_split': [2, 5, 10]
}

rf_base = RandomForestRegressor(random_state=42, n_jobs=-1)

# GridSearchCV con validación cruzada de 5 folds, optimizando para R² (por defecto en regresión)
grid_rf = GridSearchCV(
    estimator=rf_base,
    param_grid=param_grid_rf,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=1
)

grid_rf.fit(X_train_scaled, y_train)

# Guardar el mejor modelo y sus predicciones
mejor_rf = grid_rf.best_estimator_
modelos_entrenados['Random Forest'] = mejor_rf
predicciones_test['Random Forest'] = mejor_rf.predict(X_test_scaled)

fin = time.time()
print(f"\n✅ Random Forest entrenado y optimizado en {fin-inicio:.2f} segundos.")
print(f"   🏆 Mejores hiperparámetros encontrados: {grid_rf.best_params_}")
print(f"   📊 Mejor R² en validación cruzada: {grid_rf.best_score_:.4f}")

# ----------------------------------------------------------
# 4.3. MODELO 3: XGBOOST CON GRIDSEARCHCV
# ----------------------------------------------------------
print("\n🔹 4.3. Entrenando XGBoost con Validación Cruzada (GridSearch)...")
inicio = time.time()

# Definir la grilla de hiperparámetros para XGBoost
param_grid_xgb = {
    'n_estimators': [50, 100, 150],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1]
}

xgb_base = XGBRegressor(random_state=42, n_jobs=-1, objective='reg:squarederror')

# GridSearchCV
grid_xgb = GridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid_xgb,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=1
)

grid_xgb.fit(X_train_scaled, y_train)

# Guardar el mejor modelo y sus predicciones
mejor_xgb = grid_xgb.best_estimator_
modelos_entrenados['XGBoost'] = mejor_xgb
predicciones_test['XGBoost'] = mejor_xgb.predict(X_test_scaled)

fin = time.time()
print(f"\n✅ XGBoost entrenado y optimizado en {fin-inicio:.2f} segundos.")
print(f"   🏆 Mejores hiperparámetros encontrados: {grid_xgb.best_params_}")
print(f"   📊 Mejor R² en validación cruzada: {grid_xgb.best_score_:.4f}")

# ----------------------------------------------------------
# 4.4. RESUMEN DE LA FASE 4
# ----------------------------------------------------------
print("\n" + "="*60)
print("✅ FASE 4 COMPLETADA - RESUMEN DE MODELOS")
print("="*60)
for nombre, modelo in modelos_entrenados.items():
    print(f"📌 {nombre}: Listo para evaluación en el conjunto de Test.")

print("\n🎯 Los modelos están entrenados y listos para la FASE 5: EVALUACIÓN Y COMPARACIÓN")

FASE 4: ENTRENAMIENTO Y OPTIMIZACIÓN DE MODELOS

🔹 4.1. Entrenando Regresión Lineal Múltiple (Baseline)...
✅ Regresión Lineal entrenada en 0.0317 segundos.
   (Nota: La regresión lineal no tiene hiperparámetros para tunear).

🔹 4.2. Entrenando Random Forest con Validación Cruzada (GridSearch)...
Fitting 5 folds for each of 36 candidates, totalling 180 fits

✅ Random Forest entrenado y optimizado en 65.94 segundos.
   🏆 Mejores hiperparámetros encontrados: {'max_depth': None, 'min_samples_split': 5, 'n_estimators': 50}
   📊 Mejor R² en validación cruzada: 0.7765

🔹 4.3. Entrenando XGBoost con Validación Cruzada (GridSearch)...
Fitting 5 folds for each of 27 candidates, totalling 135 fits

✅ XGBoost entrenado y optimizado en 7.76 segundos.
   🏆 Mejores hiperparámetros encontrados: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 50}
   📊 Mejor R² en validación cruzada: 0.7622

✅ FASE 4 COMPLETADA - RESUMEN DE MODELOS
📌 Regresión Lineal: Listo para evaluación en el conjunto de Test.